# RAG reporting evidence extraction for Section 4.5

This notebook **does not modify** `rag_pipeline.py` or `report_generator.py`.  
It only wraps the existing pipeline and generator to export the data needed for writing **4.5 RAG-Assisted Reporting Results**.

It produces three main outputs:

1. **Case-level summary table** for representative cases such as TP, TN, FP, FN, and Borderline.
2. **Full draft report texts** with evidence IDs and safety checks.
3. **Fixed reporting constraints summary** taken directly from the payload.

## Important note
The fields **`image_retrieval_support`** and **`text_retrieval_support`** are **wrapper-level summary labels** created in this notebook for writing convenience.  
They do **not** change the original pipeline logic.


In [1]:

from pathlib import Path
import sys
import json
import math
import traceback
import pandas as pd
import numpy as np

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "rag_pipeline.py").exists() and (p / "report_generator.py").exists():
            return p
    raise FileNotFoundError(
        "Could not find project root containing rag_pipeline.py and report_generator.py. "
        "Place this notebook inside the project folder or edit PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# ---------------------------
# Edit these parameters
# ---------------------------
from ui_constants import MODEL_VARIANTS

print("Available MODEL_VARIANTS keys:", list(MODEL_VARIANTS.keys()))

VARIANT_KEY = "baseline_lsmf"   # <-- change if needed
ASSETS_DIR = PROJECT_ROOT / "GUI" / "assets" / VARIANT_KEY
REPORT_DIR = PROJECT_ROOT / "rag_reporting_eval_outputs" / VARIANT_KEY

TOPK_IMG = 5
TOPK_TEXT = 6

# Prefer test-like splits first. If not found, the notebook falls back to all rows.
PREFERRED_SPLITS = {"test", "val", "validation"}

# Number of examples to export for each role.
SAMPLE_PER_ROLE = 1

# If metadata columns are incomplete, the notebook can fall back to scanning images.
MAX_FALLBACK_SCAN = 200

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ASSETS_DIR =", ASSETS_DIR)
print("REPORT_DIR =", REPORT_DIR)


FileNotFoundError: Could not find project root containing rag_pipeline.py and report_generator.py. Place this notebook inside the project folder or edit PROJECT_ROOT manually.

In [ ]:

from rag_pipeline import PneumoRAGPipeline
from report_generator import ReportGenerator, evidence_ok, unsupported_detail_flags, word_count_en

pipeline = PneumoRAGPipeline(
    variant_key=VARIANT_KEY,
    assets_dir=ASSETS_DIR,
    report_dir=REPORT_DIR,
)

generator = ReportGenerator()

print("Loaded variant:", pipeline.variant_name)
print("Expected embedding dim:", pipeline.expected_embedding_dim)
print("Metadata rows:", len(pipeline.meta))
print("Threshold:", pipeline.THR)
print("Temperature T:", pipeline.T)
print("Path column:", pipeline.PATH_COL)
print("Case ID column:", pipeline.CASEID_COL)
print("Split column:", pipeline.SPLIT_COL)
print("Y true column:", pipeline.YTRUE_COL)
print("Probability column:", pipeline.PCOL)
print("Y pred column:", pipeline.YPRED_COL)


## Helper functions

The helpers below do four things:

- resolve usable image paths from metadata
- choose representative cases from metadata when possible
- run the unchanged pipeline and generator on selected cases
- export summary tables and full records


In [ ]:

def resolve_case_id(row: pd.Series, pipeline) -> str:
    if pipeline.CASEID_COL and pipeline.CASEID_COL in row.index:
        return str(row[pipeline.CASEID_COL])
    raw_path = row[pipeline.PATH_COL]
    return Path(str(raw_path)).name

def resolve_image_path_from_row(row: pd.Series, pipeline) -> str | None:
    raw_path = row[pipeline.PATH_COL]
    return pipeline.resolve_existing_image_path(raw_path)

def preferred_split_mask(df: pd.DataFrame, pipeline) -> pd.Series:
    if pipeline.SPLIT_COL is None or pipeline.SPLIT_COL not in df.columns:
        return pd.Series([True] * len(df), index=df.index)

    split_lower = df[pipeline.SPLIT_COL].astype(str).str.lower()
    preferred = split_lower.isin({s.lower() for s in PREFERRED_SPLITS})
    if preferred.any():
        return preferred
    return pd.Series([True] * len(df), index=df.index)

def image_support_label_from_payload(payload: dict) -> str:
    ctx = (((payload or {}).get("image_rag") or {}).get("behaviour_context") or {})
    num_cases = int(ctx.get("num_cases") or 0)
    retrieval_state = str(ctx.get("retrieval_state") or "limited")
    low_similarity = bool(ctx.get("low_similarity", False))

    if num_cases <= 0:
        return "No"
    if retrieval_state == "agreement" and not low_similarity:
        return "Yes"
    return "Partial"

def text_support_label_from_payload(payload: dict) -> str:
    chunks = (((payload or {}).get("text_rag") or {}).get("evidence_chunks") or [])
    n = len(chunks)
    if n <= 0:
        return "No"
    if n >= 3:
        return "Yes"
    return "Partial"

def make_confusion_role(y_true, y_pred) -> str:
    if pd.isna(y_true) or pd.isna(y_pred):
        return "Unknown"
    y_true = int(y_true)
    y_pred = int(y_pred)
    if y_true == 1 and y_pred == 1:
        return "TP"
    if y_true == 0 and y_pred == 0:
        return "TN"
    if y_true == 0 and y_pred == 1:
        return "FP"
    if y_true == 1 and y_pred == 0:
        return "FN"
    return "Unknown"

def add_metadata_prediction_fields(df: pd.DataFrame, pipeline) -> pd.DataFrame:
    out = df.copy()

    if pipeline.PCOL and pipeline.PCOL in out.columns:
        out["p_meta"] = pd.to_numeric(out[pipeline.PCOL], errors="coerce")
        out["margin_meta"] = (out["p_meta"] - float(pipeline.THR)).abs()
        out["confidence_band_meta"] = out["p_meta"].apply(
            lambda p: pipeline.confidence_band(float(p), float(pipeline.THR)) if pd.notna(p) else None
        )
    else:
        out["p_meta"] = np.nan
        out["margin_meta"] = np.nan
        out["confidence_band_meta"] = None

    if pipeline.YPRED_COL and pipeline.YPRED_COL in out.columns:
        out["y_pred_meta"] = pd.to_numeric(out[pipeline.YPRED_COL], errors="coerce")
    else:
        out["y_pred_meta"] = np.nan

    if pipeline.YTRUE_COL and pipeline.YTRUE_COL in out.columns:
        out["y_true_meta"] = pd.to_numeric(out[pipeline.YTRUE_COL], errors="coerce")
    else:
        out["y_true_meta"] = np.nan

    out["confusion_role_meta"] = [
        make_confusion_role(yt, yp) for yt, yp in zip(out["y_true_meta"], out["y_pred_meta"])
    ]
    return out

def shortlist_from_metadata(pipeline, sample_per_role: int = 1) -> pd.DataFrame:
    df = pipeline.meta.copy()
    df = df[preferred_split_mask(df, pipeline)].copy()

    df["resolved_image_path"] = df.apply(lambda row: resolve_image_path_from_row(row, pipeline), axis=1)
    df = df[df["resolved_image_path"].notna()].copy()

    df = add_metadata_prediction_fields(df, pipeline)

    has_required = (
        df["y_true_meta"].notna().any()
        and df["y_pred_meta"].notna().any()
        and df["p_meta"].notna().any()
    )
    if not has_required:
        raise RuntimeError("Metadata does not contain enough columns for direct representative selection.")

    chosen = []
    used_idx = set()

    role_sorting = {
        "TP": ("p_meta", False),   # highest positive confidence
        "TN": ("p_meta", True),    # lowest negative confidence
        "FP": ("p_meta", False),   # strongest false positive
        "FN": ("p_meta", True),    # strongest false negative
    }

    for role, (sort_col, ascending) in role_sorting.items():
        block = df[df["confusion_role_meta"] == role].sort_values(sort_col, ascending=ascending)
        block = block[~block.index.isin(used_idx)]
        if len(block) == 0:
            continue
        take = block.head(sample_per_role).copy()
        take["selected_role"] = role
        chosen.append(take)
        used_idx.update(take.index.tolist())

    # Borderline cases are selected by smallest margin to threshold.
    border = df.sort_values("margin_meta", ascending=True)
    border = border[~border.index.isin(used_idx)]
    if len(border) > 0:
        take = border.head(sample_per_role).copy()
        take["selected_role"] = "Borderline"
        chosen.append(take)
        used_idx.update(take.index.tolist())

    if not chosen:
        raise RuntimeError("No representative cases could be selected from metadata.")

    out = pd.concat(chosen, axis=0).reset_index(drop=True)
    out["case_id"] = out.apply(lambda row: resolve_case_id(row, pipeline), axis=1)
    return out

def fallback_scan_shortlist(pipeline, sample_per_role: int = 1, max_scan: int = 200) -> pd.DataFrame:
    """
    Fallback route if metadata does not contain y_true/y_pred/p columns.
    This route is slower because it must call build_payload on candidate images.
    """
    rows = []
    df = pipeline.meta.copy()
    df = df[preferred_split_mask(df, pipeline)].copy()
    df["resolved_image_path"] = df.apply(lambda row: resolve_image_path_from_row(row, pipeline), axis=1)
    df = df[df["resolved_image_path"].notna()].copy()

    scanned = 0
    for _, row in df.iterrows():
        if scanned >= max_scan:
            break

        image_path = row["resolved_image_path"]
        try:
            payload = pipeline.build_payload(image_path, topk_img=TOPK_IMG, topk_text=TOPK_TEXT)
            pred = payload["prediction"]
            y_pred = int(pred["y_pred"])
            p = float(pred["p_calibrated"])
            margin = abs(p - float(pred["threshold"]))
            y_true = None
            if pipeline.YTRUE_COL and pipeline.YTRUE_COL in row.index:
                try:
                    y_true = int(row[pipeline.YTRUE_COL])
                except Exception:
                    y_true = None

            rows.append({
                "case_id": resolve_case_id(row, pipeline),
                "resolved_image_path": image_path,
                "y_true_meta": y_true,
                "y_pred_meta": y_pred,
                "p_meta": p,
                "margin_meta": margin,
                "confidence_band_meta": pred["confidence_band"],
                "confusion_role_meta": make_confusion_role(y_true, y_pred) if y_true is not None else "Unknown",
                "selected_role": None,
            })
            scanned += 1

        except Exception as e:
            print(f"[scan skip] {image_path}: {e}")

    scan_df = pd.DataFrame(rows)
    if len(scan_df) == 0:
        raise RuntimeError("Fallback scan produced no usable rows.")

    chosen = []
    used_idx = set()

    role_sorting = {
        "TP": ("p_meta", False),
        "TN": ("p_meta", True),
        "FP": ("p_meta", False),
        "FN": ("p_meta", True),
    }

    if "confusion_role_meta" in scan_df.columns:
        for role, (sort_col, ascending) in role_sorting.items():
            block = scan_df[scan_df["confusion_role_meta"] == role].sort_values(sort_col, ascending=ascending)
            block = block[~block.index.isin(used_idx)]
            if len(block) == 0:
                continue
            take = block.head(sample_per_role).copy()
            take["selected_role"] = role
            chosen.append(take)
            used_idx.update(take.index.tolist())

    border = scan_df.sort_values("margin_meta", ascending=True)
    border = border[~border.index.isin(used_idx)]
    if len(border) > 0:
        take = border.head(sample_per_role).copy()
        take["selected_role"] = "Borderline"
        chosen.append(take)
        used_idx.update(take.index.tolist())

    if not chosen:
        raise RuntimeError("Fallback scan could not select any representative cases.")

    return pd.concat(chosen, axis=0).reset_index(drop=True)

def run_full_case(image_path: str, row_like: dict | pd.Series, pipeline, generator) -> dict:
    report, payload, out_path = generator.generate_pneumo_report(
        pipeline=pipeline,
        image_path=image_path,
        topk_img=TOPK_IMG,
        topk_text=TOPK_TEXT,
        save=True,
    )

    pred = payload["prediction"]
    img_ctx = payload["image_rag"]["behaviour_context"]
    text_chunks = payload["text_rag"]["evidence_chunks"]
    report_text = str(report.get("diagnostic_report", "")).strip()

    flags = unsupported_detail_flags(report_text)
    unsupported_claim = any(flags.values())
    ev_ok = evidence_ok(report, payload)
    fail_safe = bool(report.get("fail_safe", False))
    rule_adherent = bool(ev_ok and (not unsupported_claim) and (not fail_safe))

    y_true = None
    if isinstance(row_like, pd.Series):
        if pipeline.YTRUE_COL and pipeline.YTRUE_COL in row_like.index:
            try:
                y_true = int(row_like[pipeline.YTRUE_COL])
            except Exception:
                y_true = None
    elif isinstance(row_like, dict):
        y_true = row_like.get("y_true_meta")

    y_pred = int(pred["y_pred"])
    confusion_role = make_confusion_role(y_true, y_pred) if y_true is not None else "Unknown"

    return {
        "selected_role": row_like.get("selected_role") if isinstance(row_like, dict) else row_like.get("selected_role"),
        "case_role": confusion_role,
        "case_id": row_like.get("case_id") if isinstance(row_like, dict) else row_like.get("case_id", resolve_case_id(row_like, pipeline)),
        "image_path": image_path,
        "y_true": y_true,
        "y_pred": y_pred,
        "p_calibrated": float(pred["p_calibrated"]),
        "threshold": float(pred["threshold"]),
        "temperature_T": float(pred["temperature_T"]),
        "confidence_band": pred["confidence_band"],
        "narrative_label": pred["narrative_label"],
        "image_retrieval_support": image_support_label_from_payload(payload),
        "image_retrieval_state": img_ctx.get("retrieval_state"),
        "image_retrieval_agreement_rate": img_ctx.get("agreement_rate"),
        "image_retrieval_mean_similarity": img_ctx.get("mean_similarity"),
        "image_retrieval_num_cases": img_ctx.get("num_cases"),
        "text_retrieval_support": text_support_label_from_payload(payload),
        "text_chunk_count": len(text_chunks),
        "report_evidence_ok": ev_ok,
        "unsupported_claim": unsupported_claim,
        "unsupported_localization": bool(flags.get("unsupported_localization", False)),
        "unsupported_size": bool(flags.get("unsupported_size", False)),
        "unsupported_tension": bool(flags.get("unsupported_tension", False)),
        "unsupported_imaging_sign": bool(flags.get("unsupported_imaging_sign", False)),
        "rule_adherent": rule_adherent,
        "fail_safe": fail_safe,
        "fail_reason": report.get("fail_reason"),
        "word_count": word_count_en(report_text),
        "diagnostic_report": report_text,
        "evidence_text_chunk_ids": json.dumps((report.get("evidence", {}) or {}).get("text_chunk_ids", []), ensure_ascii=False),
        "evidence_retrieved_case_ids": json.dumps((report.get("evidence", {}) or {}).get("retrieved_case_ids", []), ensure_ascii=False),
        "payload_text_chunk_ids": json.dumps([c["chunk_id"] for c in text_chunks], ensure_ascii=False),
        "payload_retrieved_case_ids": json.dumps(payload["image_rag"].get("retrieved_case_ids", []), ensure_ascii=False),
        "report_json_path": out_path,
        "payload_obj": payload,
        "report_obj": report,
    }


In [ ]:

# Try direct metadata-based selection first.
try:
    selected_df = shortlist_from_metadata(pipeline, sample_per_role=SAMPLE_PER_ROLE)
    selection_mode = "metadata"
except Exception as e:
    print("Metadata-based selection failed:", e)
    print("Falling back to slower scan mode ...")
    selected_df = fallback_scan_shortlist(
        pipeline,
        sample_per_role=SAMPLE_PER_ROLE,
        max_scan=MAX_FALLBACK_SCAN,
    )
    selection_mode = "fallback_scan"

print("Selection mode:", selection_mode)
display_cols = [
    c for c in [
        "selected_role",
        "case_id",
        "resolved_image_path",
        "y_true_meta",
        "y_pred_meta",
        "p_meta",
        "margin_meta",
        "confidence_band_meta",
        "confusion_role_meta",
    ] if c in selected_df.columns
]
selected_df[display_cols]


## Run the unchanged generator on selected cases

This step generates the actual draft reports only for the selected representative cases.


In [ ]:

full_records = []

for _, row in selected_df.iterrows():
    image_path = row["resolved_image_path"]
    print(f"Running case: {row['selected_role']} | {row['case_id']}")
    try:
        rec = run_full_case(image_path, row, pipeline, generator)
        full_records.append(rec)
    except Exception as e:
        print(f"[failed] {image_path}: {e}")
        traceback.print_exc()

if len(full_records) == 0:
    raise RuntimeError("No full records were generated.")

results_df = pd.DataFrame([
    {k: v for k, v in rec.items() if k not in {"payload_obj", "report_obj"}}
    for rec in full_records
])

results_df


In [ ]:

# Export outputs for writing Section 4.5
summary_csv = REPORT_DIR / "rag_reporting_summary.csv"
full_jsonl = REPORT_DIR / "rag_reporting_full_records.jsonl"
constraints_json = REPORT_DIR / "rag_reporting_constraints_summary.json"
reports_txt = REPORT_DIR / "rag_reporting_reports_preview.txt"

results_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

with open(full_jsonl, "w", encoding="utf-8") as f:
    for rec in full_records:
        obj = {
            k: v for k, v in rec.items()
            if k not in {"payload_obj", "report_obj"}
        }
        obj["payload"] = rec["payload_obj"]
        obj["report"] = rec["report_obj"]
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

constraint_payload = full_records[0]["payload_obj"]
constraint_obj = {
    "task": constraint_payload.get("task"),
    "report_constraints": constraint_payload.get("report_constraints", {}),
    "system_prompt_rules_source": "report_generator.py SYSTEM + payload.report_constraints",
    "notes": {
        "allow_localization": False,
        "allow_laterality": False,
        "allow_size": False,
        "allow_extent": False,
        "allow_tension": False,
        "allow_specific_imaging_signs": False,
    }
}
with open(constraints_json, "w", encoding="utf-8") as f:
    json.dump(constraint_obj, f, ensure_ascii=False, indent=2)

with open(reports_txt, "w", encoding="utf-8") as f:
    for rec in full_records:
        f.write("=" * 80 + "\n")
        f.write(f"selected_role: {rec['selected_role']}\n")
        f.write(f"case_role: {rec['case_role']}\n")
        f.write(f"case_id: {rec['case_id']}\n")
        f.write(f"p_calibrated: {rec['p_calibrated']:.6f}\n")
        f.write(f"report_evidence_ok: {rec['report_evidence_ok']}\n")
        f.write(f"unsupported_claim: {rec['unsupported_claim']}\n")
        f.write(f"rule_adherent: {rec['rule_adherent']}\n")
        f.write(f"fail_safe: {rec['fail_safe']}\n")
        f.write("diagnostic_report:\n")
        f.write(rec["diagnostic_report"] + "\n\n")

print("Saved:", summary_csv)
print("Saved:", full_jsonl)
print("Saved:", constraints_json)
print("Saved:", reports_txt)


## Clean writing table for Section 4.5

This table is the one most directly useful for the report text.


In [ ]:

writing_cols = [
    "selected_role",
    "case_role",
    "case_id",
    "y_true",
    "y_pred",
    "p_calibrated",
    "confidence_band",
    "narrative_label",
    "image_retrieval_support",
    "text_retrieval_support",
    "report_evidence_ok",
    "unsupported_claim",
    "rule_adherent",
    "fail_safe",
    "diagnostic_report",
]
results_df[writing_cols]


In [ ]:

# Optional: simple aggregate checks for quick narrative support
agg = {
    "n_cases": int(len(results_df)),
    "report_evidence_ok_count": int(results_df["report_evidence_ok"].sum()),
    "unsupported_claim_count": int(results_df["unsupported_claim"].sum()),
    "rule_adherent_count": int(results_df["rule_adherent"].sum()),
    "fail_safe_count": int(results_df["fail_safe"].sum()),
    "mean_word_count": float(results_df["word_count"].mean()),
}

agg_df = pd.DataFrame([agg])
agg_df


## Notes for writing

Use the exported files in `REPORT_DIR`:

- `rag_reporting_summary.csv` for the case-level table
- `rag_reporting_full_records.jsonl` for complete payload + report objects
- `rag_reporting_constraints_summary.json` for fixed safety constraints
- `rag_reporting_reports_preview.txt` for quick reading of final report texts

If FP or FN cases are not found, increase `SAMPLE_PER_ROLE` or `MAX_FALLBACK_SCAN`, or switch to a split with more errors.
